# Notebook 11 — Formal $M_1^{\star}$ vs $M_3$ comparison (reviewer point 3.2)

$M_3$ is Wolf$(\alpha, \beta)$ + linear residual (5 parameters). It is non-nested with respect to $M_1^{\star}$ (2 parameters), so a likelihood-ratio / F-test is not directly applicable. We report:

* the non-nested comparison via AIC/BIC,
* the Cox / Vuong-style non-nested comparison (difference in log-likelihoods with standard error over observations),
* an F-test of the nested sub-question: does $(\alpha, \beta)$ on top of $M_1^{\star}$ improve the fit? (i.e. $M_1^{\star}$ vs. the full $M_3 = $ Wolf$(\alpha,\beta) + $ linear).

Writes `paper-rev-01/rev01_m1_vs_m3.json`.

In [1]:
import math, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import integrate, optimize, stats as sstats

WORK = Path.cwd()
DATA_DIR = WORK / 'ml_data'
OUT_DIR = WORK / 'paper-rev-01'
C2_TWIN = 0.6601618158468695739278121100145557784326233602847334133194484233354

def hl_correct(g):
    if g <= 0 or g % 2: return 0.0
    n = g
    while n % 2 == 0: n //= 2
    corr, p = 1.0, 3
    while p*p <= n:
        if n % p == 0:
            corr *= (p-1)/(p-2)
            while n % p == 0: n //= p
        p += 2
    if n > 1: corr *= (n-1)/(n-2)
    return 2*C2_TWIN*corr

def omega(n):
    if n <= 1: return 0
    c, d = 0, 2
    while d*d <= n:
        if n % d == 0:
            c += 1
            while n % d == 0: n //= d
        d += 1
    if n > 1: c += 1
    return c

def li2(N):
    v, _ = integrate.quad(lambda t: 1.0/np.log(t)**2, 2.0, N, limit=200)
    return float(v)

cached = sorted(DATA_DIR.glob('gaps_N*.csv'))
histograms = {int(f.stem.replace('gaps_N','')): pd.read_csv(f) for f in cached}
rows = []
for N, df in histograms.items():
    lnN=math.log(N); lnlnN=math.log(lnN); li2_N=li2(N)
    for _, r in df.iterrows():
        g, emp = int(r['gap']), int(r['count'])
        if g<2 or g>100 or g%2 or emp<100: continue
        Cg = hl_correct(g); rho = g*Cg/lnN
        if not (0.05 <= rho <= 1.10): continue
        rows.append({'N':N,'g':g,'log_empir':math.log(emp),
                     'log_C_li':math.log(Cg*li2_N),'rho':rho,
                     'sqrt_omega_g':math.sqrt(omega(g)),'lnlnN':lnlnN,
                     'weight':min(1.0,emp/1000.0)})
D = pd.DataFrame(rows)
print(f'n = {len(D)}')

n = 140


In [2]:
y = D['log_empir'].values
xC = D['log_C_li'].values
rho = D['rho'].values
sqw = D['sqrt_omega_g'].values
lnlnN = D['lnlnN'].values
w = D['weight'].values
n = len(D)

def weighted_rss(resid, w):
    return float(np.sum(w*resid**2))

def loglik_and_ic(rss_w, k, n):
    sigma2 = rss_w / n
    ll = -0.5*n*(math.log(2*math.pi*sigma2) + 1.0)
    return ll, 2*k - 2*ll, k*math.log(n) - 2*ll

# --- M1_star: 2 params (a, b), no intercept ---
X = np.column_stack([sqw, lnlnN])
target = y - (xC - rho)
ws = np.sqrt(w)
coefs_1s, *_ = np.linalg.lstsq(X*ws[:,None], target*ws, rcond=None)
resid_1s = target - X @ coefs_1s
rss_1s = weighted_rss(resid_1s, w)
k_1s = 2 + 1  # +sigma^2
ll_1s, aic_1s, bic_1s = loglik_and_ic(rss_1s, k_1s, n)

# --- M3: Wolf(alpha, beta) + a*sqrt(omega) + b*lnlnN + c (5 params) ---
# Model: yhat = xC - alpha*rho^beta + a*sqw + b*lnlnN + c
def neg_rss_M3(theta):
    alpha, beta, a, b, c = theta
    yhat = xC - alpha*(rho**beta) + a*sqw + b*lnlnN + c
    return float(np.sum(w*(y - yhat)**2))

res3 = optimize.minimize(neg_rss_M3, x0=[0.42, 1.64, 0.5, -0.1, 0.0],
                         method='Nelder-Mead',
                         options={'xatol':1e-8, 'fatol':1e-10, 'maxiter':20000})
alpha3, beta3, a3, b3, c3 = res3.x
rss_3 = float(res3.fun)
k_3 = 5 + 1
ll_3, aic_3, bic_3 = loglik_and_ic(rss_3, k_3, n)

# --- Intermediate nested model: fix (alpha, beta) = (1, 1) (=> Wolf) + linear w/o intercept = M1_star ---
# M3 nests M1_star when (alpha, beta, c) = (1, 1, 0). So a standard LR test of M1_star vs M3 is valid.
LR = 2*(ll_3 - ll_1s)
df = 5 - 2
p_LR = 1 - sstats.chi2.cdf(LR, df=df)
# F-test equivalent (Gaussian small-sample): F = ((RSS1-RSS3)/q) / (RSS3/(n-p_full))
q = df
p_full = 5
F = ((rss_1s - rss_3)/q) / (rss_3/(n - p_full))
p_F = 1 - sstats.f.cdf(F, q, n - p_full)

# --- Vuong-style non-nested comparison ---
# Per-observation log-likelihoods (Gaussian, MLE sigma each model)
sigma2_1s = rss_1s/n
sigma2_3  = rss_3/n
li_1s = -0.5*(math.log(2*math.pi*sigma2_1s) + w*resid_1s**2/sigma2_1s)
resid_3 = y - (xC - alpha3*(rho**beta3) + a3*sqw + b3*lnlnN + c3)
li_3  = -0.5*(math.log(2*math.pi*sigma2_3)  + w*resid_3**2/sigma2_3)
diff = li_3 - li_1s
mean_diff = float(diff.mean()); std_diff = float(diff.std(ddof=1))
vuong = mean_diff * math.sqrt(n) / std_diff if std_diff > 0 else float('nan')
p_vuong = 2*(1 - sstats.norm.cdf(abs(vuong)))

print('=== fits ===')
print(f'M1*  (a, b)           = ({coefs_1s[0]:+.4f}, {coefs_1s[1]:+.4f})  RSS_w={rss_1s:.4f}  AIC={aic_1s:+.3f}  BIC={bic_1s:+.3f}')
print(f'M3   (a, b, a, b, c)  = alpha={alpha3:+.4f}, beta={beta3:+.4f}, a={a3:+.4f}, b={b3:+.4f}, c={c3:+.4f}')
print(f'                       RSS_w={rss_3:.4f}  AIC={aic_3:+.3f}  BIC={bic_3:+.3f}')
print()
print('=== nested test (M1* nests in M3 at alpha=1, beta=1, c=0) ===')
print(f'LR chi^2({df}) = {LR:.3f}   p = {p_LR:.4f}')
print(f'F({q}, {n-p_full}) = {F:.3f}   p = {p_F:.4f}')
print()
print('=== Vuong-style non-nested ===')
print(f'mean(log lik_M3 - log lik_M1*) = {mean_diff:+.6f}, std = {std_diff:.4f}')
print(f'Vuong z = {vuong:+.3f}, two-sided p = {p_vuong:.4f}')
print()
print('=== Delta AIC / BIC ===')
print(f'AIC(M3) - AIC(M1*) = {aic_3 - aic_1s:+.3f}')
print(f'BIC(M3) - BIC(M1*) = {bic_3 - bic_1s:+.3f}')

=== fits ===
M1*  (a, b)           = (+0.8552, -0.2017)  RSS_w=1.1114  AIC=-273.739  BIC=-264.915
M3   (a, b, a, b, c)  = alpha=+0.6889, beta=+1.1667, a=+0.5483, b=-0.1834, c=+0.0991
                       RSS_w=0.4826  AIC=-384.523  BIC=-366.873

=== nested test (M1* nests in M3 at alpha=1, beta=1, c=0) ===
LR chi^2(3) = 116.783   p = 0.0000
F(3, 135) = 58.630   p = 0.0000

=== Vuong-style non-nested ===
mean(log lik_M3 - log lik_M1*) = +0.417082, std = 0.6674
Vuong z = +7.394, two-sided p = 0.0000

=== Delta AIC / BIC ===
AIC(M3) - AIC(M1*) = -110.783
BIC(M3) - BIC(M1*) = -101.958


In [3]:
out = {
    'n': n,
    'M1_star': {'coefs': coefs_1s.tolist(), 'rss_w': rss_1s,
                'loglik': ll_1s, 'aic': aic_1s, 'bic': bic_1s, 'k': k_1s},
    'M3':      {'coefs': [float(alpha3), float(beta3), float(a3), float(b3), float(c3)],
                'rss_w': rss_3, 'loglik': ll_3, 'aic': aic_3, 'bic': bic_3, 'k': k_3},
    'nested_LR': {'stat': float(LR), 'df': df, 'p_value': float(p_LR)},
    'F_test':    {'F': float(F), 'df1': q, 'df2': n - p_full, 'p_value': float(p_F)},
    'vuong':     {'z': float(vuong), 'p_value': float(p_vuong),
                  'mean_diff': mean_diff, 'std_diff': std_diff},
    'delta_ic':  {'dAIC_M3_minus_M1s': aic_3 - aic_1s,
                  'dBIC_M3_minus_M1s': bic_3 - bic_1s},
}
(OUT_DIR / 'rev01_m1_vs_m3.json').write_text(json.dumps(out, indent=2))
print('Saved:', OUT_DIR / 'rev01_m1_vs_m3.json')

Saved: /opt/apps/jupyter/work/paper-rev-01/rev01_m1_vs_m3.json
